# TUẦN 1: 2.1 + 2.2

In [ ]:
# Cài Java 11 (khuyến nghị cho Spark 3.5.x)
!apt-get update -qq
!apt-get install openjdk-11-jdk-headless -qq > /dev/null

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

print("Java version:")
!java -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Java version:
openjdk version "17.0.18" 2026-01-20
OpenJDK Runtime Environment (build 17.0.18+8-Ubuntu-122.04.1)
OpenJDK 64-Bit Server VM (build 17.0.18+8-Ubuntu-122.04.1, mixed mode, sharing)


In [ ]:
# Cách 2: Tải Spark binary thủ công
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz

import os
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"
!pip install findspark -q
import findspark
findspark.init()

In [ ]:
# Khởi tạo SparkSession
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("Olist_Preprocessing_Week1_Final") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()
print("Spark Session đã khởi tạo thành công!")

Spark Session đã khởi tạo thành công!


In [ ]:
!pip install -q streamlit
print("\nStreamlit installed successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 87.2 MB/s eta 0:00:00

Streamlit installed successfully.


In [ ]:
!python --version

Python 3.12.13


In [ ]:
!pip install -q gradio

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
base_path = "/content/drive/MyDrive/datasetCuoiKi/data/"

In [ ]:
# Đọc dữ liệu
orders_df = spark.read.csv(base_path + "olist_orders_dataset.csv", header=True, inferSchema=True)
customers_df = spark.read.csv(base_path + "olist_customers_dataset.csv", header=True, inferSchema=True)
order_items_df = spark.read.csv(base_path + "olist_order_items_dataset.csv", header=True, inferSchema=True)
payments_df = spark.read.csv(base_path + "olist_order_payments_dataset.csv", header=True, inferSchema=True)
reviews_df = spark.read.csv(base_path + "olist_order_reviews_dataset.csv", header=True, inferSchema=True)
products_df = spark.read.csv(base_path + "olist_products_dataset.csv", header=True, inferSchema=True)
sellers_df = spark.read.csv(base_path + "olist_sellers_dataset.csv", header=True, inferSchema=True)
category_df = spark.read.csv(base_path + "product_category_name_translation.csv", header=True, inferSchema=True)
geolocation_df = spark.read.csv(base_path + "olist_geolocation_dataset.csv", header=True, inferSchema=True)
print("Đã đọc xong 9 file CSV")

Đã đọc xong 9 file CSV


In [ ]:
from pyspark.sql.functions import col, sum

def explore_full(df, name):
    print("\n==============================")
    print("DATASET:", name)
    print("==============================")

    # 1. Số dòng
    total_rows = df.count()
    print("Số dòng:", total_rows)

    # 2. Schema
    print("\nSchema:")
    df.printSchema()

    # 3. Dữ liệu mẫu
    print("\n5 dòng đầu:")
    df.show(5, truncate=False)

    # 4. Thống kê mô tả
    print("\nThống kê:")
    df.describe().show()

    # 5. Kiểm tra null
    print("\nSố lượng NULL theo cột:")
    null_df = df.select([
        sum(col(c).isNull().cast("int")).alias(c)
        for c in df.columns
    ])
    null_df.show()

    # 6. Kiểm tra duplicate
    unique_rows = df.dropDuplicates().count()
    print("\nDuplicate:")
    print("Tổng:", total_rows)
    print("Unique:", unique_rows)
    print("Số dòng trùng:", total_rows - unique_rows)

In [ ]:
explore_full(customers_df, "customers")


DATASET: customers
Số dòng: 99441

Schema:
root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)


5 dòng đầu:
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|customer_id                     |customer_unique_id              |customer_zip_code_prefix|customer_city        |customer_state|
+--------------------------------+--------------------------------+------------------------+---------------------+--------------+
|06b8999e2fba1a1fbc88172c00ba8bc7|861eff4711a542e4b93843c6dd7febb0|14409                   |franca               |SP            |
|18955e83d337fd6b2def6b18a428ac77|290c77bc529b7ac935b93aa66c333dc3|9790                    |sao bernardo do campo|SP            |
|4e7b3e00288586ebd08712fdd0374a03|060e732b5b29

In [ ]:
explore_full(geolocation_df, "geolocation")


DATASET: geolocation
Số dòng: 1000163

Schema:
root
 |-- geolocation_zip_code_prefix: integer (nullable = true)
 |-- geolocation_lat: double (nullable = true)
 |-- geolocation_lng: double (nullable = true)
 |-- geolocation_city: string (nullable = true)
 |-- geolocation_state: string (nullable = true)


5 dòng đầu:
+---------------------------+-------------------+------------------+----------------+-----------------+
|geolocation_zip_code_prefix|geolocation_lat    |geolocation_lng   |geolocation_city|geolocation_state|
+---------------------------+-------------------+------------------+----------------+-----------------+
|1037                       |-23.54562128115268 |-46.63929204800168|sao paulo       |SP               |
|1046                       |-23.546081127035535|-46.64482029837157|sao paulo       |SP               |
|1046                       |-23.54612896641469 |-46.64295148361138|sao paulo       |SP               |
|1041                       |-23.5443921648681  |-46.63949

In [ ]:
explore_full(order_items_df, "order_items")


DATASET: order_items
Số dòng: 112650

Schema:
root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)


5 dòng đầu:
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|order_id                        |order_item_id|product_id                      |seller_id                       |shipping_limit_date|price|freight_value|
+--------------------------------+-------------+--------------------------------+--------------------------------+-------------------+-----+-------------+
|00010242fe8c5a6d1ba2dd792cb16214|1            |4244733e06e7ecb4970a6e2683c13e61|48436dade18ac8b2bce089ec2a041202|2017-09-19 09:45:35|58.9 |13.29        |
|00018f77

In [ ]:
explore_full(payments_df, "payments")


DATASET: payments
Số dòng: 103886

Schema:
root
 |-- order_id: string (nullable = true)
 |-- payment_sequential: integer (nullable = true)
 |-- payment_type: string (nullable = true)
 |-- payment_installments: integer (nullable = true)
 |-- payment_value: double (nullable = true)


5 dòng đầu:
+--------------------------------+------------------+------------+--------------------+-------------+
|order_id                        |payment_sequential|payment_type|payment_installments|payment_value|
+--------------------------------+------------------+------------+--------------------+-------------+
|b81ef226f3fe1789b1e8b2acac839d17|1                 |credit_card |8                   |99.33        |
|a9810da82917af2d9aefd1278f1dcfa0|1                 |credit_card |1                   |24.39        |
|25e8ea4e93396b6fa0d3dd708e76c1bd|1                 |credit_card |1                   |65.71        |
|ba78997921bbcdc1373bb41e913ab953|1                 |credit_card |8                   |107.7

In [ ]:
explore_full(reviews_df, "reviews")


DATASET: reviews
Số dòng: 104162

Schema:
root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: string (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: string (nullable = true)
 |-- review_answer_timestamp: string (nullable = true)


5 dòng đầu:
+--------------------------------+--------------------------------+------------+--------------------+----------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                              |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+---

In [ ]:
explore_full(orders_df, "orders")


DATASET: orders
Số dòng: 99441

Schema:
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)


5 dòng đầu:
+--------------------------------+--------------------------------+------------+------------------------+-------------------+----------------------------+-----------------------------+-----------------------------+
|order_id                        |customer_id                     |order_status|order_purchase_timestamp|order_approved_at  |order_delivered_carrier_date|order_delivered_customer_date|order_estimated_delivery_date|
+--------------------------------+--------------------------------+------------+-

In [ ]:
explore_full(products_df, "products")


DATASET: products
Số dòng: 32951

Schema:
root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)


5 dòng đầu:
+--------------------------------+---------------------+-------------------+--------------------------+------------------+----------------+-----------------+-----------------+----------------+
|product_id                      |product_category_name|product_name_lenght|product_description_lenght|product_photos_qty|product_weight_g|product_length_cm|product_height_cm|product_width_cm|
+--------------------------------+---------------------+-------------------+---------------------

In [ ]:
explore_full(sellers_df, "sellers")


DATASET: sellers
Số dòng: 3095

Schema:
root
 |-- seller_id: string (nullable = true)
 |-- seller_zip_code_prefix: integer (nullable = true)
 |-- seller_city: string (nullable = true)
 |-- seller_state: string (nullable = true)


5 dòng đầu:
+--------------------------------+----------------------+-----------------+------------+
|seller_id                       |seller_zip_code_prefix|seller_city      |seller_state|
+--------------------------------+----------------------+-----------------+------------+
|3442f8959a84dea7ee197c632cb2df15|13023                 |campinas         |SP          |
|d1b65fc7debc3361ea86b5f14c68d2e2|13844                 |mogi guacu       |SP          |
|ce3ad9de960102d0677a81f5d0bb7b2d|20031                 |rio de janeiro   |RJ          |
|c0f3eea2e14555b6faeea3dd58c1b1c3|4195                  |sao paulo        |SP          |
|51a04a8a6bdcb23deccc82b0b80742cf|12914                 |braganca paulista|SP          |
+--------------------------------+-----------

In [ ]:
explore_full(category_df, "category_translation")


DATASET: category_translation
Số dòng: 71

Schema:
root
 |-- product_category_name: string (nullable = true)
 |-- product_category_name_english: string (nullable = true)


5 dòng đầu:
+----------------------+-----------------------------+
|product_category_name |product_category_name_english|
+----------------------+-----------------------------+
|beleza_saude          |health_beauty                |
|informatica_acessorios|computers_accessories        |
|automotivo            |auto                         |
|cama_mesa_banho       |bed_bath_table               |
|moveis_decoracao      |furniture_decor              |
+----------------------+-----------------------------+
only showing top 5 rows


Thống kê:
+-------+---------------------+-----------------------------+
|summary|product_category_name|product_category_name_english|
+-------+---------------------+-----------------------------+
|  count|                   71|                           71|
|   mean|                 NULL|     

In [ ]:
from pyspark.sql.functions import *

In [ ]:
# Geolocation
geo_customers = geolocation_df.select(
    col("geolocation_zip_code_prefix").alias("customer_zip_code_prefix"),
    col("geolocation_city").alias("customer_geo_city"),
    col("geolocation_state").alias("customer_geo_state")
).dropDuplicates()

customers_df = customers_df.join(geo_customers, "customer_zip_code_prefix", "left")

geo_sellers = geolocation_df.select(
    col("geolocation_zip_code_prefix").alias("seller_zip_code_prefix"),
    col("geolocation_city").alias("seller_geo_city"),
    col("geolocation_state").alias("seller_geo_state")
).dropDuplicates()

sellers_df = sellers_df.join(geo_sellers, "seller_zip_code_prefix", "left")

In [ ]:
# Timestamp & Cast
orders_df = orders_df \
    .withColumn("order_purchase_timestamp", try_to_timestamp(col("order_purchase_timestamp"))) \
    .withColumn("order_approved_at", try_to_timestamp(col("order_approved_at"))) \
    .withColumn("order_delivered_carrier_date", try_to_timestamp(col("order_delivered_carrier_date"))) \
    .withColumn("order_delivered_customer_date", try_to_timestamp(col("order_delivered_customer_date"))) \
    .withColumn("order_estimated_delivery_date", try_to_timestamp(col("order_estimated_delivery_date")))

reviews_df = reviews_df \
    .withColumn("review_creation_date", try_to_timestamp(col("review_creation_date"))) \
    .withColumn("review_answer_timestamp", try_to_timestamp(col("review_answer_timestamp")))

order_items_df = order_items_df \
    .withColumn("shipping_limit_date", try_to_timestamp(col("shipping_limit_date"))) \
    .withColumn("price", col("price").cast("double")) \
    .withColumn("freight_value", col("freight_value").cast("double"))

payments_df = payments_df.withColumn("payment_value", col("payment_value").cast("double"))

from pyspark.sql.functions import expr

reviews_df = reviews_df \
    .withColumn("review_score", expr("try_cast(review_score as int)"))
#reviews_df = reviews_df.withColumn("review_score", col("review_score").cast("int"))

In [ ]:
# Aggregate
order_items_agg = order_items_df.groupBy("order_id").agg(
    first("product_id").alias("product_id"),
    first("seller_id").alias("seller_id"),
    sum("price").alias("total_price"),
    sum("freight_value").alias("total_freight_value"),
    count("order_item_id").alias("num_items")
)

payments_agg = payments_df.groupBy("order_id").agg(
    sum("payment_value").alias("total_payment_value"),
    first("payment_type").alias("payment_type")
)

In [ ]:
# JOIN
df_master = orders_df.alias("o") \
    .join(customers_df.alias("c"), col("o.customer_id") == col("c.customer_id"), "left") \
    .drop(col("c.customer_id")) \
    .join(order_items_agg.alias("oi"), col("o.order_id") == col("oi.order_id"), "left") \
    .drop(col("oi.order_id")) \
    .join(payments_agg.alias("op"), col("o.order_id") == col("op.order_id"), "left") \
    .drop(col("op.order_id")) \
    .join(reviews_df.alias("r"), col("o.order_id") == col("r.order_id"), "left") \
    .drop(col("r.order_id")) \
    .join(products_df.alias("p"), col("oi.product_id") == col("p.product_id"), "left") \
    .drop(col("p.product_id")) \
    .join(sellers_df.alias("s"), col("oi.seller_id") == col("s.seller_id"), "left") \
    .drop(col("s.seller_id")) \
    .join(category_df.alias("t"), col("p.product_category_name") == col("t.product_category_name"), "left") \
    .drop(col("t.product_category_name"))

In [ ]:
# ================================
# XỬ LÝ MISSING VALUES (FINAL) - ĐÃ SỬA
# ================================

# 1. Tạo cột delivery_delay_days trước (nếu chưa có)
from pyspark.sql.functions import datediff, col

df_master = df_master.withColumn(
    "delivery_delay_days",
    datediff(
        col("order_delivered_customer_date"),
        col("order_estimated_delivery_date")
    )
)

# 2. Xử lý missing values
df_master = df_master.fillna({
    # TEXT
    "review_comment_message": "",
    "review_comment_title": "",

    # CATEGORICAL
    "payment_type": "unknown",
    "product_category_name_english": "unknown",

    # NUMERICAL
    "delivery_delay_days": 0,      # giờ đã tồn tại
    "num_items": 1,

    # NUMERICAL (bổ sung)
    "total_payment_value": 0,
    "total_price": 0,
    "total_freight_value": 0
})

# Drop rows thiếu key quan trọng
df_master = df_master.dropna(subset=["order_id"])

print("✅ Missing values đã xử lý đầy đủ")

✅ Missing values đã xử lý đầy đủ


In [ ]:
# Feature Engineering
df_master = df_master \
    .withColumn("delivery_delay_days", datediff(col("order_delivered_customer_date"), col("order_estimated_delivery_date"))) \
    .withColumn("total_order_value", col("total_price") + col("total_freight_value")) \
    .withColumn("is_late_delivery", when(col("order_delivered_customer_date") > col("order_estimated_delivery_date"), 1).otherwise(0))

# ÉP 1 DÒNG CHO 1 ORDER_ID
df_master = df_master.dropDuplicates(["order_id"])

# THANH HUYỀN

In [ ]:
# ===================== IV.6.A USER-ITEM MATRIX =====================

from pyspark.sql.functions import count

user_item_df = df.select("customer_unique_id","product_id","order_id") \
    .groupBy("customer_unique_id","product_id") \
    .agg(count("*").alias("purchase_count"))

print("User-Item pairs:", user_item_df.count())
user_item_df.show(10)

User-Item pairs: 98255
+--------------------+--------------------+--------------+
|  customer_unique_id|          product_id|purchase_count|
+--------------------+--------------------+--------------+
|0bff960830c81727b...|3324aa94229088467...|             1|
|a976f7c2d2b94b928...|b76364870c38ddd7d...|             1|
|e3b7fece1ec0c35e0...|0bbdc963004d9b2fd...|             1|
|18f769ac0da309db8...|5b3242867b3c1f86e...|             1|
|1bbd9a63db6ed2bb1...|a54244559e62c8ef2...|             1|
|0ad16341f17f16908...|34a7ef14a5e7901f4...|             1|
|745fe69e5cc0a523b...|3d0552899dbe2f89c...|             1|
|408cb3e09282d6705...|7340a3839a1de1e99...|             1|
|f27c05c74dfcba909...|dc858768341433779...|             1|
|f4b32085bb03ac03a...|8bb7efbc6db04c5cc...|             1|
+--------------------+--------------------+--------------+
only showing top 10 rows



In [ ]:
# ===================== IV.6.B INDEX IDs =====================

from pyspark.ml.feature import StringIndexer

user_indexer = StringIndexer(
    inputCol="customer_unique_id",
    outputCol="user_id"
).fit(user_item_df)

item_indexer = StringIndexer(
    inputCol="product_id",
    outputCol="item_id"
).fit(user_item_df)

als_ready_df = user_indexer.transform(user_item_df)
als_ready_df = item_indexer.transform(als_ready_df)

als_ready_df.select(
    "customer_unique_id","product_id",
    "user_id","item_id","purchase_count"
).show(5)

+--------------------+--------------------+-------+-------+--------------+
|  customer_unique_id|          product_id|user_id|item_id|purchase_count|
+--------------------+--------------------+-------+-------+--------------+
|0bff960830c81727b...|3324aa94229088467...| 6988.0|16780.0|             1|
|a976f7c2d2b94b928...|b76364870c38ddd7d...|64088.0| 1048.0|             1|
|e3b7fece1ec0c35e0...|0bbdc963004d9b2fd...|85280.0|  372.0|             1|
|18f769ac0da309db8...|5b3242867b3c1f86e...|11701.0| 6088.0|             1|
|1bbd9a63db6ed2bb1...|a54244559e62c8ef2...|12683.0| 1525.0|             1|
+--------------------+--------------------+-------+-------+--------------+
only showing top 5 rows



In [ ]:
# ===================== IV.6.C ALS MODEL =====================

from pyspark.ml.recommendation import ALS

train_data, test_data = als_ready_df.randomSplit([0.8, 0.2], seed=42)

als = ALS(
    userCol="user_id",
    itemCol="item_id",
    ratingCol="purchase_count",
    implicitPrefs=True,
    rank=20,
    maxIter=15,
    regParam=0.1,
    alpha=1.0,
    coldStartStrategy="drop"
)

als_model = als.fit(train_data)
print("ALS model trained.")

ALS model trained.


In [ ]:
# ===================== IV.6.E RANKING METRICS =====================

from pyspark.sql.functions import collect_set, col
from pyspark.mllib.evaluation import RankingMetrics

true_items = test_data.groupBy("user_id") \
    .agg(collect_set("item_id").alias("true_items"))

pred_items = user_recs.select(
    "user_id",
    col("recommendations.item_id").alias("pred_items")
)

ranking_df = pred_items.join(true_items, on="user_id")

ranking_rdd = ranking_df.rdd.map(
    lambda row: (row.pred_items, row.true_items)
)

metrics = RankingMetrics(ranking_rdd)

print("Precision:", metrics.precisionAt(10))
print("Recall:", metrics.recallAt(10))
print("MAP:", metrics.meanAveragePrecision)

/content/spark-3.5.1-bin-hadoop3/python/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Precision: 0.0027348394768133178
Recall: 0.02734839476813318
MAP: 0.014410754392918489


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, collect_set
from pyspark.ml.feature import StringIndexer
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.fpm import FPGrowth

In [ ]:
# Load master processed
df = spark.read.parquet("/content/drive/MyDrive/datasetCuoiKi/data/olist_master_processed.parquet")

df = df.select(
    "customer_unique_id",
    "product_id",
    "order_id"
).dropna()


In [ ]:
# ===== Implicit feedback  =====
# số lần mua product của customer
als_df = df.groupBy("customer_unique_id", "product_id") \
           .agg(count("*").alias("purchase_count"))



In [ ]:
# ===== Encode user, item =====
user_indexer = StringIndexer(inputCol="customer_unique_id", outputCol="user_id", handleInvalid="skip")
item_indexer = StringIndexer(inputCol="product_id", outputCol="item_id", handleInvalid="skip")

als_df = user_indexer.fit(als_df).transform(als_df)
als_df = item_indexer.fit(als_df).transform(als_df)


In [ ]:
# ===== Train / Test split =====
train_data, test_data = als_df.randomSplit([0.8, 0.2], seed=42)

In [ ]:
# ===== Train ALS (implicit) =====
als = ALS(
    userCol="user_id",
    itemCol="item_id",
    ratingCol="purchase_count",
    implicitPrefs=True,
    rank=20,
    maxIter=15,
    regParam=0.1,
    alpha=1.0,
    coldStartStrategy="drop"
)

model_als = als.fit(train_data)


In [ ]:
# ===== Generate recommendations =====
user_recs = model_als.recommendForAllUsers(10)
user_recs.show(5, truncate=False)

+-------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|user_id|recommendations                                                                                                                                                                                                   |
+-------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|1      |[{18, 4.338871E-7}, {21, 3.817467E-7}, {30, 3.1218335E-7}, {35, 3.1034293E-7}, {14, 2.7904466E-7}, {17, 2.6329343E-7}, {29, 2.3366205E-7}, {11, 2.1111599E-7}, {36, 1.923276E-7}, {16, 1.743837E-7}]              |
|3      |[{37, 2.7200033E-13}, {22, 1.8943405E-13}, {49, 1.6678283E-13}, {27, 1.6211518E-13}, {34, 1.5584746E-13}, {

In [ ]:
from pyspark.sql.functions import col, collect_set
from pyspark.mllib.evaluation import RankingMetrics

In [ ]:
# ===================== ALS EVALUATION WITH RANKING METRICS =====================

from pyspark.sql.functions import col, collect_set
from pyspark.mllib.evaluation import RankingMetrics

# Ground truth: sản phẩm user đã mua trong test set
true_items = test_data.groupBy("user_id") \
    .agg(collect_set("item_id").alias("true_items"))

# Predicted items từ ALS
pred_items = user_recs.select(
    "user_id",
    col("recommendations.item_id").alias("pred_items")
)

# Join predicted vs ground truth
ranking_df = pred_items.join(true_items, on="user_id")

# Convert sang RDD đúng format (pred, true)
ranking_rdd = ranking_df.rdd.map(lambda row: (row.pred_items, row.true_items))

metrics = RankingMetrics(ranking_rdd)

print("Precision:", metrics.precisionAt(10))
print("Recall:", metrics.recallAt(10))
print("MAP:", metrics.meanAveragePrecision)

/content/spark-3.5.1-bin-hadoop3/python/pyspark/sql/context.py:158: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Precision: 0.0027348394768133178
Recall: 0.02734839476813318
MAP: 0.014410754392918489


**2. FP-GROWTH**

In [ ]:
from pyspark.sql.functions import collect_set, size

In [ ]:
# ===================== IV.6.F FP TRANSACTIONS =====================



df_fp = order_items.join(products, "product_id") \
    .select("order_id","product_category_name")

transactions = df_fp.groupBy("order_id") \
    .agg(collect_set("product_category_name").alias("items")) \
    .filter(size("items") > 1)

print("Transactions:", transactions.count())
transactions.show(5, truncate=False)

Transactions: 727
+--------------------------------+-----------------------------------------------------------------------+
|order_id                        |items                                                                  |
+--------------------------------+-----------------------------------------------------------------------+
|014405982914c2cde2796ddcf0b8703d|[perfumaria, esporte_lazer]                                            |
|01cce1175ac3c4a450e3a0f856d02734|[papelaria, ferramentas_jardim]                                        |
|02beac5428ccbd30f4b3fbded058289f|[papelaria, informatica_acessorios]                                    |
|0362e923f805ae4dce58fd78c86c96c4|[utilidades_domesticas, moveis_cozinha_area_de_servico_jantar_e_jardim]|
|0420dbc50fc554e303e4b2d6b39063f6|[bebes, cool_stuff]                                                    |
+--------------------------------+-----------------------------------------------------------------------+
only showing top 5 

In [ ]:
# ===================== IV.6.G FP MODEL =====================

from pyspark.ml.fpm import FPGrowth

fp = FPGrowth(
    itemsCol="items",
    minSupport=0.001,
    minConfidence=0.2
)

fp_model = fp.fit(transactions)
print("FP-Growth model trained.")

FP-Growth model trained.


In [ ]:
# ===================== IV.6.H FP RESULTS =====================

freq_items = fp_model.freqItemsets
rules = fp_model.associationRules

print("Frequent Itemsets:")
freq_items.show(10, truncate=False)

print("Association Rules:")
rules.show(10, truncate=False)

Frequent Itemsets:
+----------------------------------------+----+
|items                                   |freq|
+----------------------------------------+----+
|[eletroportateis]                       |6   |
|[eletroportateis, eletronicos]          |1   |
|[eletroportateis, utilidades_domesticas]|1   |
|[eletroportateis, beleza_saude]         |1   |
|[eletroportateis, telefonia]            |1   |
|[eletroportateis, esporte_lazer]        |1   |
|[eletroportateis, moveis_decoracao]     |1   |
|[climatizacao]                          |2   |
|[climatizacao, informatica_acessorios]  |1   |
|[climatizacao, esporte_lazer]           |1   |
+----------------------------------------+----+
only showing top 10 rows

Association Rules:
+----------------------------------------------------+--------------------+-------------------+------------------+---------------------+
|antecedent                                          |consequent          |confidence         |lift              |support      

In [ ]:
from pyspark.sql.functions import collect_set, size, col
from pyspark.ml.fpm import FPGrowth

# Load RAW data, KHÔNG dùng master_processed
order_items = spark.read.csv("/content/drive/MyDrive/datasetCuoiKi/data/olist_order_items_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv("/content/drive/MyDrive/datasetCuoiKi/data/olist_products_dataset.csv", header=True, inferSchema=True)

# Join để lấy category cho từng product trong order
df_fp = order_items.join(products, on="product_id", how="inner") \
                   .select("order_id", "product_category_name") \
                   .dropna()

# Transaction = mỗi order gồm nhiều CATEGORY
transactions = df_fp.groupBy("order_id") \
    .agg(collect_set("product_category_name").alias("items"))

transactions = transactions.filter(size("items") > 1)

print("Số transactions:", transactions.count())
transactions.show(5, truncate=False)


Số transactions: 727
+--------------------------------+-----------------------------------------------------------------------+
|order_id                        |items                                                                  |
+--------------------------------+-----------------------------------------------------------------------+
|014405982914c2cde2796ddcf0b8703d|[perfumaria, esporte_lazer]                                            |
|01cce1175ac3c4a450e3a0f856d02734|[papelaria, ferramentas_jardim]                                        |
|02beac5428ccbd30f4b3fbded058289f|[papelaria, informatica_acessorios]                                    |
|0362e923f805ae4dce58fd78c86c96c4|[utilidades_domesticas, moveis_cozinha_area_de_servico_jantar_e_jardim]|
|0420dbc50fc554e303e4b2d6b39063f6|[bebes, cool_stuff]                                                    |
+--------------------------------+-----------------------------------------------------------------------+
only showing top

In [ ]:

fp = FPGrowth(
    itemsCol="items",
    minSupport=0.001,
    minConfidence=0.2
)

model_fp = fp.fit(transactions)

model_fp.freqItemsets.orderBy(col("freq").desc()).show(10, False)
model_fp.associationRules.orderBy(col("lift").desc()).show(10, False)


+-----------------------------------+----+
|items                              |freq|
+-----------------------------------+----+
|[moveis_decoracao]                 |203 |
|[cama_mesa_banho]                  |198 |
|[utilidades_domesticas]            |102 |
|[bebes]                            |93  |
|[ferramentas_jardim]               |73  |
|[beleza_saude]                     |70  |
|[cama_mesa_banho, moveis_decoracao]|70  |
|[esporte_lazer]                    |67  |
|[cool_stuff]                       |66  |
|[informatica_acessorios]           |51  |
+-----------------------------------+----+
only showing top 10 rows

+-----------------------------------------+-----------------------------+----------+------------------+--------------------+
|antecedent                               |consequent                   |confidence|lift              |support             |
+-----------------------------------------+-----------------------------+----------+------------------+-------------------

In [ ]:
import os

base_path = "/content/drive/MyDrive/models"
os.makedirs(base_path, exist_ok=True)

als_path = f"{base_path}/als_model"
fp_path  = f"{base_path}/fp_model"

als_path, fp_path

('/content/drive/MyDrive/model/als_model',
 '/content/drive/MyDrive/model/fp_model')

In [ ]:
# ===================== SAVE ALS MODEL =====================
als_model.write().overwrite().save(als_path)
print("ALS model saved at:", als_path)

ALS model saved at: /content/drive/MyDrive/model/als_model


In [ ]:
# ===================== SAVE FP-GROWTH MODEL =====================
fp_model.write().overwrite().save(fp_path)
print("FP model saved at:", fp_path)

FP model saved at: /content/drive/MyDrive/model/fp_model
